In [1]:
import pandas as pd
import ast
import os
import warnings

warnings.filterwarnings("ignore")

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
data_dir = r"/active-data/datasets/IMG_PR/Cus_PR/IMG_VR_2023-08-08_1"
plas_data = pd.read_csv(f"{data_dir}/IMGPR_plasmid_data.tsv", sep='\t')
plas_taxon = {}
for genus_name in keep_genus:
    temp_data = plas_data[plas_data['host_taxonomy'].str.contains(f'g__{genus_name}', na=False)]
    plas_taxon[genus_name] = temp_data['plasmid_id'].to_list()

In [3]:
from Bio import SeqIO
from tqdm import tqdm

target_dir = r"/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid"
genus_plas_file = {}
for genus_name in keep_genus:
    genus_dir = f"{target_dir}/{genus_name}"
    os.makedirs(genus_dir, exist_ok=True)
    genus_plas_file[genus_name] = open(f"{genus_dir}/IMGPR_nucl.fasta", "w+")

data_dir = r"/active-data/datasets/IMG_PR/Cus_PR/IMG_VR_2023-08-08_1"
nucl_file = f"{data_dir}/IMGPR_nucl.fna"
with tqdm(total = len(plas_data), desc=f'Extract plasmids', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
    with open(nucl_file, 'r') as handle:
        seq_records = SeqIO.parse(handle, 'fasta')
        for record in seq_records:
            for genus_name in keep_genus:
                if record.id.split('|')[0] in plas_taxon[genus_name]:
                    SeqIO.write(record, genus_plas_file[genus_name], "fasta")
            pbar.update(1)

for fh in genus_plas_file.values():
    fh.close()

Extract plasmids: 100%|████████████████████████████████████████████| 700k/700k [28:53<00:00, 404B/s]
